In [1]:
!pip install -q langchain langchain-groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.1 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [5]:
"""
AI Feedback Response Generator — Gradio UI
---------------------------------------------
Classifies customer feedback (positive / negative / neutral) using a
LangChain RunnableBranch, then generates an appropriate AI response.

Setup:
    pip install gradio langchain langchain-groq langchain-core

    Set your Groq API key as an environment variable before running:
        export GROQ_API_KEY="your_key_here"      (macOS/Linux)
        setx GROQ_API_KEY "your_key_here"         (Windows)

Run:
    python feedback_response_app.py
"""

import os
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda

# ---------------------------------------------------------------------------
# LangChain setup
# ---------------------------------------------------------------------------

MODEL_NAME = "openai/gpt-oss-120b"

_model = None


def get_model():
    """Lazily build the model so a missing API key doesn't crash app startup."""
    global _model
    if _model is None:
        _model = ChatGroq(
            model=MODEL_NAME,
            groq_api_key=os.getenv("GROQ_API_KEY"),
        )
    return _model


classification_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a classifier. Classify the customer feedback as exactly one "
            "word: 'positive', 'negative', or 'neutral'. Respond with only that word.",
        ),
        ("human", "Feedback: {feedback}"),
    ]
)

positive_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a warm customer support agent. Thank the customer sincerely "
            "for their positive feedback and encourage them to keep sharing their experience.",
        ),
        ("human", "Feedback: {feedback}"),
    ]
)

negative_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an empathetic customer support agent. Apologize for the "
            "customer's negative experience, acknowledge their concern, and offer "
            "a clear next step to resolve the issue.",
        ),
        ("human", "Feedback: {feedback}"),
    ]
)

neutral_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful customer support agent. Thank the customer for "
            "their feedback and politely ask a clarifying follow-up question to "
            "better understand their experience.",
        ),
        ("human", "Feedback: {feedback}"),
    ]
)


def build_chain():
    model = get_model()
    parser = StrOutputParser()

    classification_chain = classification_template | model | parser

    branch = RunnableBranch(
        (lambda x: "positive" in x["sentiment"].lower(), positive_template | model | parser),
        (lambda x: "negative" in x["sentiment"].lower(), negative_template | model | parser),
        neutral_template | model | parser,  # default
    )

    full_chain = (
        RunnableLambda(lambda x: {"feedback": x["feedback"], "sentiment": classification_chain.invoke(x)})
        | branch
    )
    return full_chain


_chain = None


def get_chain():
    global _chain
    if _chain is None:
        _chain = build_chain()
    return _chain


# ---------------------------------------------------------------------------
# Core inference function
# ---------------------------------------------------------------------------

def analyze_feedback(feedback: str):
    if not feedback or not feedback.strip():
        return "⚠️ Please enter some feedback first."

    if not os.getenv("GROQ_API_KEY"):
        return (
            "⚠️ No GROQ_API_KEY found in your environment.\n\n"
            "Set it with:\n"
            "  export GROQ_API_KEY=\"your_key_here\"   (macOS/Linux)\n"
            "  setx GROQ_API_KEY \"your_key_here\"      (Windows)\n"
            "then restart the app."
        )

    try:
        chain = get_chain()
        return chain.invoke({"feedback": feedback.strip()})
    except Exception as e:
        return f"❌ An error occurred:\n\n{e}"


def clear_fields():
    return "", ""


# ---------------------------------------------------------------------------
# Gradio UI
# ---------------------------------------------------------------------------

CUSTOM_CSS = """
.gradio-container {max-width: 1100px !important; margin: auto;}
footer {visibility: hidden}

#header-banner {
    background: linear-gradient(135deg, #f97316 0%, #ea580c 50%, #dc2626 100%);
    border-radius: 16px;
    padding: 24px 26px;
    margin-bottom: 18px;
    box-shadow: 0 8px 24px rgba(234, 88, 12, 0.25);
}
#header-banner h1 {
    color: #ffffff !important;
    font-size: 1.6rem;
    margin: 0 0 6px 0;
    text-align: center;
}
#header-banner p {
    color: #ffedd5 !important;
    text-align: center;
    margin: 0;
    font-size: 0.92rem;
}

#input-card, #output-card {
    background: var(--block-background-fill);
    border: 1px solid var(--border-color-primary);
    border-radius: 14px;
    padding: 18px 20px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.05);
}

#generate-btn {
    background: linear-gradient(135deg, #f97316, #dc2626) !important;
    color: white !important;
    border: none !important;
    font-weight: 600 !important;
    font-size: 1.05rem !important;
    border-radius: 10px !important;
    box-shadow: 0 4px 14px rgba(220, 38, 38, 0.35) !important;
    transition: transform 0.15s ease, box-shadow 0.15s ease !important;
}
#generate-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 18px rgba(220, 38, 38, 0.45) !important;
}

#clear-btn {
    border-radius: 10px !important;
    font-weight: 500 !important;
    border: 1px solid var(--border-color-primary) !important;
}

#feedback-box textarea, #output-box textarea {
    font-size: 1.02rem !important;
    line-height: 1.7 !important;
}

#footer-note {
    text-align: center;
    color: #9ca3af;
    font-size: 0.85rem;
    margin-top: 18px;
}
"""

THEME = gr.themes.Soft(
    primary_hue="orange",
    secondary_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui"],
)

with gr.Blocks(title="AI Feedback Response Generator") as demo:

    with gr.Row(equal_height=True):
        with gr.Column(scale=1, min_width=380, elem_id="input-card"):
            gr.HTML(
                """
                <div id="header-banner">
                    <h1>📝 AI Feedback Response Generator</h1>
                    <p>Classifies customer feedback and crafts the right reply — powered by LangChain + Groq</p>
                </div>
                """
            )

            gr.Markdown("### 💬 Customer Feedback")

            feedback_input = gr.Textbox(
                lines=8,
                label="",
                show_label=False,
                placeholder="Enter customer feedback here...",
                elem_id="feedback-box",
            )

            with gr.Row():
                clear_btn = gr.Button("🗑️ Clear", elem_id="clear-btn", scale=1)
                generate_btn = gr.Button(
                    "🤖 Generate Response", elem_id="generate-btn", variant="primary", scale=2
                )

            gr.Examples(
                examples=[
                    ["The delivery was super fast and the product quality exceeded my expectations!"],
                    ["I ordered two weeks ago and still haven't received my package. Very disappointed."],
                    ["The product is okay, but the packaging could be improved."],
                ],
                inputs=[feedback_input],
                label="Try an example",
            )

        with gr.Column(scale=1, min_width=380, elem_id="output-card"):
            gr.Markdown("### 🤖 AI Response")
            output_box = gr.Textbox(
                lines=17,
                label="",
                show_label=False,
                interactive=False,
                placeholder="The AI-generated response will appear here...",
                elem_id="output-box",
            )

    gr.HTML(f"<div id='footer-note'>Powered by LangChain + Groq ({MODEL_NAME})</div>")

    generate_btn.click(
        fn=analyze_feedback,
        inputs=[feedback_input],
        outputs=output_box,
    )

    feedback_input.submit(
        fn=analyze_feedback,
        inputs=[feedback_input],
        outputs=output_box,
    )

    clear_btn.click(
        fn=clear_fields,
        inputs=[],
        outputs=[feedback_input, output_box],
    )


if __name__ == "__main__":
    demo.launch(theme=THEME, css=CUSTOM_CSS, debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ef8589ca2b89604794.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ef8589ca2b89604794.gradio.live
